# 04 · SQL storage and SQL analysis
**Step 6 & 10 (SQL part):** create the database from the ER design, load the cleaned data,
apply primary/foreign keys, verify, then answer business questions with SQL
(SELECT/WHERE/ORDER BY, GROUP BY/HAVING, joins, subqueries, CTEs, window functions, aggregations).

Connection settings come from `.env` (copy `.env.example`). Nothing is hardcoded.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()
print("Connected:", engine.url.render_as_string(hide_password=True))


def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

## Create the database and tables (`sql/schema.sql`)

In [ ]:
from src.db import create_mysql_database_if_missing
from src.load_to_sql import create_schema, load_tables, verify_load

create_mysql_database_if_missing()     # no-op for SQLite
create_schema(engine)
print("Schema created")

## Load cleaned data (from `data/cleaned/`) and verify

In [ ]:
cleaned = {t: pd.read_csv(config.CLEAN_DIR / f"{t}.csv",
                          dtype={c: "string" for c in config.STRING_COLUMNS.get(t, [])},
                          keep_default_na=True)
           for t in config.TABLE_ORDER}
# empty comments/titles come back as NaN, which SQL stores as NULL
load_tables(engine, cleaned)
display(verify_load(engine, cleaned))

## Business questions answered in SQL
Each query below uses base tables only. The same queries appear on the dashboard's
**SQL Analysis** page (`streamlit/queries.py`).

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("c2i_queries", ROOT / "streamlit" / "queries.py")
q = importlib.util.module_from_spec(spec); spec.loader.exec_module(q)

for name, spec_ in q.ANALYSIS_QUERIES.items():
    print(f"\n### {spec_['title']}\nConcepts: {', '.join(spec_['concepts'])}")
    print(spec_["sql"].strip())
    display(query(spec_["sql"]).head(15))